---
title: "Prepare 2026 course data"
format: html
editor: source
execute:
  eval: false
  warning: false
  message: false
editor_options:
  chunk_output_type: console
jupyter: ir
---

## Overview

This document downloads and prepares the public datasets used for the 2026
course update. The course focuses on paired spatial assays from sample P2 CRC:

- Visium HD, Human Colon Cancer, sample P2 CRC
- Xenium In Situ, Human Colon Cancer, sample P2 CRC
- Chromium count matrix for use as a single-cell reference during annotation

The `Space Ranger` and Xenium analysis reports are kept in the compact course
folder under `data/Human_Colon_Cancer_P2/reports/`. Large raw inputs such as
FASTQs are intentionally not downloaded here, because the course starts from
the public processed outputs.

The Visium HD and Xenium samples are serial sections. The subsets therefore
represent approximately matching tissue regions, not exact pixel-wise
registration. During preparation, overview plots are written to
`data/prep_reports/` so the ROI coordinates below can be checked if needed.

## Setup

In [ ]:
options(timeout = 3600)

library(SpatialExperiment)
library(SpatialFeatureExperiment)
library(VisiumIO)
library(DropletUtils)
library(ggplot2)
library(ggspavis)
library(HDF5Array)
library(scuttle)

find_project_root <- function(path = getwd()) {
  path <- normalizePath(path, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(path, "_quarto.yml"))) {
      return(path)
    }
    parent <- dirname(path)
    if (identical(parent, path)) {
      stop("Could not find project root containing _quarto.yml")
    }
    path <- parent
  }
}

project_root <- find_project_root()
data_path <- function(...) file.path(project_root, "data", ...)

dir.create(data_path("raw"), showWarnings = FALSE, recursive = TRUE)
dir.create(data_path("extracted"), showWarnings = FALSE, recursive = TRUE)
dir.create(data_path("prep_reports"), showWarnings = FALSE, recursive = TRUE)
dir.create(data_path("Human_Colon_Cancer_P2"), showWarnings = FALSE, recursive = TRUE)

download_if_missing <- function(url, destfile) {
  dir.create(dirname(destfile), showWarnings = FALSE, recursive = TRUE)
  size <- suppressWarnings(file.info(destfile)$size)
  if (!file.exists(destfile) || is.na(size) || size == 0) {
    message("Downloading: ", basename(destfile))
    download.file(url, destfile = destfile, mode = "wb", method = "curl")
  } else {
    message("Already present: ", destfile)
  }
}

download_if_missing_or_incomplete <- function(url, destfile, expected_size = NA_real_) {
  dir.create(dirname(destfile), showWarnings = FALSE, recursive = TRUE)
  current_size <- if (file.exists(destfile)) file.info(destfile)$size else 0

  if (!is.na(expected_size) && current_size == expected_size) {
    message("Already present with expected size: ", destfile)
    return(invisible(destfile))
  }

  if (is.na(expected_size) && current_size > 0) {
    message("Already present: ", destfile)
    return(invisible(destfile))
  }

  message("Downloading/resuming: ", basename(destfile))
  status <- system2(
    "curl",
    args = c(
      "--location",
      "--fail",
      "--continue-at", "-",
      "--retry", "10",
      "--retry-delay", "20",
      "--output", destfile,
      url
    )
  )

  if (!identical(status, 0L)) {
    stop("Download failed for: ", url)
  }

  if (!is.na(expected_size)) {
    final_size <- file.info(destfile)$size
    if (!identical(as.numeric(final_size), as.numeric(expected_size))) {
      stop(
        "Downloaded file has unexpected size: ", destfile,
        "\nExpected: ", expected_size,
        "\nObserved: ", final_size,
        "\nRerun this chunk to resume."
      )
    }
  }

  invisible(destfile)
}

extract_tar_if_missing <- function(tarfile, exdir, marker) {
  if (!file.exists(marker) && !dir.exists(marker)) {
    message("Extracting: ", basename(tarfile))
    dir.create(exdir, showWarnings = FALSE, recursive = TRUE)
    untar(tarfile, exdir = exdir)
  } else {
    message("Already extracted: ", marker)
  }
}

unzip_if_missing <- function(zipfile, exdir, marker) {
  if (!file.exists(marker) && !dir.exists(marker)) {
    message("Unzipping: ", basename(zipfile))
    dir.create(exdir, showWarnings = FALSE, recursive = TRUE)
    unzip(zipfile, exdir = exdir)
  } else {
    message("Already unzipped: ", marker)
  }
}

find_xenium_outs <- function(root) {
  candidates <- c(
    file.path(root, "outs"),
    list.dirs(root, recursive = TRUE, full.names = TRUE)
  )
  candidates <- unique(candidates)
  hits <- candidates[file.exists(file.path(candidates, "experiment.xenium"))]
  if (length(hits) == 0) {
    stop("Could not find a Xenium outs directory containing experiment.xenium under: ", root)
  }
  hits[[1]]
}

subset_by_roi <- function(x, roi) {
  xy <- spatialCoords(x)
  keep <- xy[, 1] >= roi[["xmin"]] & xy[, 1] <= roi[["xmax"]] &
    xy[, 2] >= roi[["ymin"]] & xy[, 2] <= roi[["ymax"]]
  x[, keep]
}

## Download public outputs

### Visium HD P2 CRC

In [ ]:
visium_base <- "https://cf.10xgenomics.com/samples/spatial-exp/3.0.0/Visium_HD_Human_Colon_Cancer_P2"

visium_files <- c(
  binned_outputs = "Visium_HD_Human_Colon_Cancer_P2_binned_outputs.tar.gz",
  spatial = "Visium_HD_Human_Colon_Cancer_P2_spatial.tar.gz",
  metrics = "Visium_HD_Human_Colon_Cancer_P2_metrics_summary.csv",
  web_summary = "Visium_HD_Human_Colon_Cancer_P2_web_summary.html",
  feature_slice = "Visium_HD_Human_Colon_Cancer_P2_feature_slice.h5"
)

visium_expected_sizes <- c(
  binned_outputs = 15886359364,
  spatial = NA_real_,
  metrics = NA_real_,
  web_summary = NA_real_,
  feature_slice = NA_real_
)

for (nm in names(visium_files)) {
  f <- visium_files[[nm]]
  download_if_missing_or_incomplete(
    url = file.path(visium_base, f),
    destfile = data_path("raw", f),
    expected_size = unname(visium_expected_sizes[[nm]])
  )
}

### Xenium P2 CRC

In [ ]:
xenium_base <- "https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE"

xenium_files <- c(
  gene_panel = "Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE_gene_panel.json",
  he_image = "Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE_he_image.ome.tif",
  he_alignment = "Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE_he_imagealignment.csv",
  analysis_summary = "Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE_analysis_summary.html",
  outs = "Xenium_V1_Human_Colon_Cancer_P2_CRC_Add_on_FFPE_outs.zip"
)

for (f in xenium_files) {
  download_if_missing(
    url = file.path(xenium_base, f),
    destfile = data_path("raw", f)
  )
}

### Chromium reference

In [ ]:
chromium_base <- "https://cf.10xgenomics.com/samples/cell-vdj/8.0.0/Human_DTC_5pv2_P5_CRC"

chromium_files <- c(
  aggregation = "Human_DTC_5pv2_P5_CRC_aggregation.csv",
  feature_reference = "Human_DTC_5pv2_P5_CRC_count_feature_reference.csv",
  count_matrix = "Human_DTC_5pv2_P5_CRC_count_filtered_feature_bc_matrix.h5"
)

for (f in chromium_files) {
  download_if_missing(
    url = file.path(chromium_base, f),
    destfile = data_path("raw", f)
  )
}

The Chromium immune profiling VDJ files are not downloaded by default because
they are not required for expression-based annotation. We can download them separately
if a TCR/clonotype exercise is added.

## Extract outputs

In [ ]:
visium_dir <- data_path("extracted", "Visium_HD_Human_Colon_Cancer_P2")
xenium_dir <- data_path("extracted", "Xenium_P2_CRC")

extract_tar_if_missing(
  tarfile = data_path("raw", visium_files[["binned_outputs"]]),
  exdir = visium_dir,
  marker = file.path(visium_dir, "binned_outputs")
)

extract_tar_if_missing(
  tarfile = data_path("raw", visium_files[["spatial"]]),
  exdir = visium_dir,
  marker = file.path(visium_dir, "spatial")
)

unzip_if_missing(
  zipfile = data_path("raw", xenium_files[["outs"]]),
  exdir = xenium_dir,
  marker = file.path(xenium_dir, "experiment.xenium")
)

xenium_outs <- find_xenium_outs(xenium_dir)

## Import objects

For Visium HD we use the 16 um binned output for a practical object size. The
2 um and 8 um binned outputs remain available in the extracted folder.

In [ ]:
spe_hd <- TENxVisiumHD(
  spacerangerOut = visium_dir,
  processing = "filtered",
  format = "h5",
  images = "lowres",
  bin_size = "016"
) |>
  import()

rownames(spe_hd) <- uniquifyFeatureNames(
  ID = rowData(spe_hd)$ID,
  names = rowData(spe_hd)$Symbol
)

spe_hd

In [ ]:
sfe_xen <- readXenium(xenium_outs, flip = "none")

rownames(sfe_xen) <- uniquifyFeatureNames(
  ID = rowData(sfe_xen)$ID,
  names = rowData(sfe_xen)$Symbol
)

sfe_xen

In [ ]:
sce_chromium <- read10xCounts(
  data_path("raw", chromium_files[["count_matrix"]]),
  col.names = TRUE
)

sce_chromium

## Overview plots for ROI selection

Use these plots to choose approximately matching Visium HD and Xenium regions.
The regions should cover comparable morphology, such as tumor/stroma boundary,
glands, lumen, or dense immune infiltrate.

In [ ]:
p_visium <- plotVisium(
  spe_hd,
  spots = TRUE,
  point_shape = 22,
  point_size = 0.2
)

ggsave(
  filename = data_path("prep_reports", "visium_hd_p2_crc_overview.png"),
  plot = p_visium,
  width = 8,
  height = 7,
  dpi = 200
)

p_visium

In [ ]:
xy_xen <- as.data.frame(spatialCoords(sfe_xen))
#names(xy_xen)[1:2] <- c("x", "y")

## We can plot the flipped cooridnates to match the Visium slide
p_xenium <- ggplot(xy_xen, aes(x = -y_centroid, y = -x_centroid)) +
  geom_point(size = 0.05, alpha = 0.3) +
  coord_fixed() +
  theme_bw() +
  labs(x = "x", y = "y")

ggsave(
  filename = data_path("prep_reports", "xenium_p2_crc_overview.png"),
  plot = p_xenium,
  width = 8,
  height = 7,
  dpi = 200
)

p_xenium

library(patchwork)
p_visium + p_xenium

## Plot the Xenium image
## The image has 4 channels, which are DAPI, ATP1A1/CD45/E-Cadherin, 18S, and AlphaSMA/Vimentin, in this order
## 3 channels have to be chosen to be plotted in the R, G, and B channels in this order.
plotImage(sfe_xen, image_id = "morphology_focus", channel = 1:3, show_axes = TRUE,
          dark = TRUE, normalize_channels = TRUE)

After flipping coordinates and inverting x and y, we get matching pictures across datasets.


## Subset approximately matching regions

The Visium HD subset uses an OSTA-style region of interest as a starting point.
This region is large enough to contain tissue structure, but small enough for
interactive teaching. The old P1 course object contained approximately 14,000
16 um bins after subsetting; for P2 we aim for the same order of magnitude,
roughly 10,000 to 20,000 bins.

The Xenium subset is chosen to be morphologically comparable to the Visium HD
region. Because Visium HD and Xenium are serial sections, this is not an
exact same-cell or same-coordinate comparison. Instead, it is a practical
region-level comparison between sequencing-based and image-based spatial data.

The coordinates below are the finalized course ROIs selected after inspecting
the overview plots. They are stored here so the course objects can be
regenerated reproducibly.

In [ ]:
# Coordinates are in the native coordinate systems returned by spatialCoords().

# OSTA uses this region for detailed visualization. In this run it contains
# about 4,700 16 um bins, which is useful for plotting but smaller than the
# old P1 course object.
osta_roi_visium <- c(
  xmin = 54000,
  xmax = 58000,
  ymin = 12000,
  ymax = 16000
)

# We therefore use a nearby, larger, morphology-rich region for the practical.
# This keeps the subset close to the OSTA area while including more tissue
# structure and a number of bins closer to the old P1 exercise.
roi_visium <- c(
  xmin = 50000,
  xmax = 58000,
  ymin = 14500,
  ymax = 22000
)

# Xenium coordinates are not in the same coordinate system as Visium HD. We use
# flip = "none" when importing Xenium so the compact course data has a stable
# native coordinate orientation even though morphology images are not included.
# This ROI was selected from the Xenium overview plot to capture a
# morphology-rich region with gland-like structures and stromal/low-density
# areas, broadly comparable to the Visium HD course region.
roi_xenium <- c(
  xmin = 2200,
  xmax = 4500,
  ymin = 2500,
  ymax = 4900
)
roi_visium
roi_xenium

## Inspect Xenium's ROI
p_xen <- plotImage(sfe_xen, image_id = "morphology_focus", channel = 1:3, show_axes = TRUE,
          dark = TRUE, normalize_channels = TRUE) +
    geom_sf(data = st_as_sfc(st_bbox(roi_xenium)), fill = NA, color = "white", linewidth = 0.5)
p_xen
plotImage(sfe_xen, image_id = "morphology_focus", channel = 3:1,
          bbox = roi_xenium, normalize_channels = TRUE)

## Inspect Visium HD's ROI
## We need to scale the coordinates using scaleFactors
roi_visium_scaled <- roi_visium * scaleFactors(spe_hd)
p_vis <- plotVisium(spe_hd, 
                    zoom = TRUE, spots = FALSE) + 
    geom_rect(
        xmin=roi_visium_scaled["xmin"], xmax=roi_visium_scaled["xmax"], 
        ymin=nrow(imgRaster(spe)) - roi_visium_scaled["ymin"], ymax=nrow(imgRaster(spe)) - roi_visium_scaled["ymax"], 
        col="white", linewidth = 0.5, fill=NA)
p_vis
## Subset: 
#p_vis_zoom <- plotVisium(spe, spots = FALSE, zoom = TRUE)
#p_vis + p_vis_zoom

## Compare Xenium 
p_vis + p_xen

## Indeed they do not match!
## Adjust ROI for Visium:
roi_visium_new <- c(
  xmin = 49000,
  xmax = 58000,
  ymin = 7500,
  ymax = 16000
)
roi_visium_scaled <- roi_visium_new * scaleFactors(spe_hd)
p_vis_new <- plotVisium(spe_hd, 
                    zoom = TRUE, spots = FALSE) + 
    geom_rect(
        xmin=roi_visium_scaled["xmin"], 
        xmax=roi_visium_scaled["xmax"], 
        ymin=nrow(imgRaster(spe_hd)) - roi_visium_scaled["ymin"], 
        ymax=nrow(imgRaster(spe_hd)) - roi_visium_scaled["ymax"], 
        col="white", linewidth = 0.5, fill=NA)
p_vis + p_vis_new
p_vis_new + p_xen
## Much better!

In [ ]:
spe_hd_p2_crc_sub <- subset_by_roi(spe_hd, roi_visium_new)
sfe_xen_p2_crc_sub <- subset_by_roi(sfe_xen, roi_xenium)

dim(spe_hd)
dim(spe_hd_p2_crc_sub)
dim(sfe_xen)
dim(sfe_xen_p2_crc_sub)

if (ncol(spe_hd_p2_crc_sub) < 10000 || ncol(spe_hd_p2_crc_sub) > 20000) {
  message(
    "The Visium HD subset contains ", ncol(spe_hd_p2_crc_sub),
    " bins. Consider adjusting roi_visium to keep the practical object ",
    "near 10,000-20,000 bins."
  )
}

if (ncol(sfe_xen_p2_crc_sub) == 0) {
  stop(
    "The Xenium subset contains 0 cells. Check roi_xenium against ",
    "range(spatialCoords(sfe_xen))."
  )
}

In [ ]:
p_visium_sub <- plotVisium(
  spe_hd_p2_crc_sub,
  spots = TRUE,
  zoom = TRUE,
  point_shape = 22,
  point_size = 0.6
)

xy_xen_sub <- as.data.frame(spatialCoords(sfe_xen_p2_crc_sub))
names(xy_xen_sub)[1:2] <- c("x", "y")

p_xenium_sub <- ggplot(xy_xen_sub, aes(x = x, y = y)) +
  geom_point(size = 0.08, alpha = 0.4) +
  coord_fixed() +
  theme_bw() +
  labs(x = "x", y = "y")

p_visium_pigr <- plotVisium(
  spe_hd_p2_crc_sub,
  annotate = "PIGR",
  assay = "counts",
  zoom = TRUE,
  point_size = 1,
  point_shape = 22
)

# PIGR is retained as a quick marker check because it shows a clear spatial
# pattern across the finalized Visium HD course subset.

ggsave(
  filename = data_path("prep_reports", "visium_hd_p2_crc_subset.png"),
  plot = p_visium_sub,
  width = 7,
  height = 6,
  dpi = 200
)

ggsave(
  filename = data_path("prep_reports", "xenium_p2_crc_subset.png"),
  plot = p_xenium_sub,
  width = 7,
  height = 6,
  dpi = 200
)

ggsave(
  filename = data_path("prep_reports", "visium_hd_p2_crc_subset_PIGR.png"),
  plot = p_visium_pigr,
  width = 7,
  height = 6,
  dpi = 200
)

p_visium_sub
p_visium_pigr
p_xenium_sub

## Assemble compact course data folder

The public 10x outputs are too large for course distribution. We therefore
assemble a compact folder that mirrors the previous course structure:
students download `Human_Colon_Cancer_P2.tar.gz`, extract it inside `data/`,
and then the exercises import and subset the data from
`data/Human_Colon_Cancer_P2/`.

In [ ]:
course_dir <- data_path("Human_Colon_Cancer_P2")
unlink(course_dir, recursive = TRUE)
dir.create(course_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(course_dir, "binned_outputs"), recursive = TRUE)
dir.create(file.path(course_dir, "xenium", "outs"), recursive = TRUE)
dir.create(file.path(course_dir, "chromium"), recursive = TRUE)
dir.create(file.path(course_dir, "reports"), recursive = TRUE)

# Visium HD: retain the filtered HDF5 matrix and spatial files for each bin
# size. This is sufficient for TENxVisiumHD(..., processing = "filtered",
# format = "h5") and allows students to inspect available bin sizes.
for (bin in c("square_002um", "square_008um", "square_016um")) {
  source_bin <- file.path(visium_dir, "binned_outputs", bin)
  target_bin <- file.path(course_dir, "binned_outputs", bin)
  dir.create(target_bin, recursive = TRUE)
  dir.create(file.path(target_bin, "spatial"), recursive = TRUE)

  file.copy(
    from = file.path(source_bin, "filtered_feature_bc_matrix.h5"),
    to = file.path(target_bin, "filtered_feature_bc_matrix.h5"),
    overwrite = TRUE
  )

  file.copy(
    from = list.files(file.path(source_bin, "spatial"), full.names = TRUE),
    to = file.path(target_bin, "spatial"),
    overwrite = TRUE,
    recursive = TRUE
  )
}

# Xenium: retain the minimum output files needed to import the object and work
# with cell/nucleus geometries in the exercise.
xenium_keep <- c(
  "experiment.xenium",
  "gene_panel.json",
  "metrics_summary.csv",
  "cell_feature_matrix.h5",
  "cells.parquet",
  "cell_boundaries.parquet",
  "nucleus_boundaries.parquet",
  "cell_boundaries_sf.parquet",
  "nucleus_boundaries_sf.parquet"
)

file.copy(
  from = file.path(xenium_outs, xenium_keep[file.exists(file.path(xenium_outs, xenium_keep))]),
  to = file.path(course_dir, "xenium", "outs"),
  overwrite = TRUE
)

# Chromium reference for annotation exercises.
file.copy(
  from = data_path("raw", chromium_files),
  to = file.path(course_dir, "chromium"),
  overwrite = TRUE
)

# Keep selected 10x reports with the compact data folder.
file.copy(
  from = data_path("raw", visium_files[c("metrics", "web_summary")]),
  to = file.path(course_dir, "reports"),
  overwrite = TRUE
)

file.copy(
  from = data_path("raw", xenium_files[["analysis_summary"]]),
  to = file.path(course_dir, "reports"),
  overwrite = TRUE
)

writeLines(
  c(
    "# Human Colon Cancer P2 course data",
    "",
    "Compact input data for the 2026 course exercises.",
    "",
    "This folder is designed to be downloaded as `Human_Colon_Cancer_P2.tar.gz` and extracted inside the project `data/` directory, producing `data/Human_Colon_Cancer_P2/`.",
    "",
    "Contents:",
    "",
    "- `binned_outputs/`: trimmed Visium HD P2 CRC Space Ranger binned outputs. The exercises use the 16 um filtered HDF5 matrix and spatial image files, while 2 um and 8 um folders are retained so students can inspect available bin sizes.",
    "- `xenium/outs/`: minimal Xenium P2 CRC output files required to import the object for the course exercise. Preprocessed `*_sf.parquet` segmentation files are included when available so the exercise loads faster.",
    "- `chromium/`: Chromium Single Cell Flex aggregated count matrix and metadata files for annotation exercises.",
    "- `reports/`: selected 10x summary reports, including the Visium metrics CSV, Visium web summary HTML, and Xenium analysis summary HTML.",
    "",
    "Course ROIs used in Exercise 1:",
    "",
    paste0("- Visium HD: ", paste(names(roi_visium), roi_visium, sep = "=", collapse = ", ")),
    paste0("- Xenium: ", paste(names(roi_xenium), roi_xenium, sep = "=", collapse = ", ")),
    "",
    "The Visium HD and Xenium sections are serial sections. The selected ROIs are morphology-rich and practical for teaching, but are not exact same-cell or same-coordinate regions."
  ),
  con = file.path(course_dir, "README.md")
)

course_subset_summary <- data.frame(
  object = c("spe_hd_p2_crc_sub", "sfe_xen_p2_crc_sub", "sce_chromium"),
  rows = c(nrow(spe_hd_p2_crc_sub), nrow(sfe_xen_p2_crc_sub), nrow(sce_chromium)),
  columns = c(ncol(spe_hd_p2_crc_sub), ncol(sfe_xen_p2_crc_sub), ncol(sce_chromium))
)

write.csv(
  course_subset_summary,
  file = data_path("prep_reports", "course_subset_summary.csv"),
  row.names = FALSE
)

## Session information

In [ ]:
sessionInfo()